# Classificação hierárquica de comentários de clientes com o Strands Decider

Classifica os comentários do gold com o **Strands Decider** (modelo de decisão local, sem API) usando a **taxonomia completa com as descrições**, e compara com os rótulos do gold.

**Fluxo**

| Etapa | O que faz | Chamadas ao Decider |
|---|---|---|
| 1. Mascaramento | Remove dados sensíveis do texto (CPF, cartão, telefone, e-mail, conta etc.) antes de qualquer coisa | — |
| 2. Assunto (macro) | *"Sobre o que o cliente está falando?"* entre as macrocategorias | 1 |
| 3. Assunto (sub) | Mesma pergunta, entre as subcategorias da macro escolhida, cada uma com a descrição | 1 (ou 2, se houver uma 2ª macro forte) |
| 4. Sentimento | *"Qual o sentimento do cliente em relação a {tema}?"*, usando a resposta da etapa 3 | 1 (as 2 categorias vão na mesma chamada) |
| 5. Avaliação | Macro, sub exata na 1ª, sub em comum, sentimento, calibração | — |

**Por que em 2 etapas (macro → sub) e não todas as subs de uma vez?** O Decider lê no máximo 4.096 tokens por chamada (texto + pergunta + opções). Com ~160 subcategorias com descrição, não cabe. A célula de orçamento de tokens mostra isso com os seus números; se couber, dá para trocar `ESTRATEGIA` para `"plana"`.

**Por que o sentimento é uma chamada separada?** As perguntas de uma mesma chamada são avaliadas de forma independente (uma não vê a resposta da outra). Para o sentimento depender do tema escolhido, ele precisa vir depois.

**Antes de rodar**, em outro terminal (com o venv ativo), deixe o servidor no ar:

```
strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --device cpu --port 8000
```

## 0. Configuração

Ajuste os caminhos e os nomes de coluna para os seus arquivos.

- **Gold**: formato longo, 1 linha por `(comment_id, ordem)`, com o texto, macro, sub e sentimento.
- **Taxonomia**: a planilha v2.3 com a aba de subcategorias (macro, sub, descrição, exemplos) e a aba `Macros (Estágio 1)` com a descrição de cada macro.

In [ ]:
%pip install pandas requests openpyxl

In [ ]:
import json, re, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# --- arquivos ---
ARQ_GOLD = "dados/exemplo_gold.xlsx"   # troque pelo seu gold (.xlsx ou .csv)
ABA_GOLD = "gold"
ARQ_TAXONOMIA = "dados/exemplo_taxonomia.xlsx"
ABA_SUBS = "Subcategorias (Estágio 2)"
ABA_MACROS = "Macros (Estágio 1)"       # None para montar a descrição da macro só com a lista de subs
PASTA_SAIDA = Path("saida_decider"); PASTA_SAIDA.mkdir(exist_ok=True)

# --- colunas do gold ---

COL_ID, COL_ORDEM, COL_TEXTO = "comment_id", "ordem", "comentario"
COL_MACRO, COL_SUB, COL_SENT = "categoria_macro", "subcategoria", "sentimento"
COL_ROTULADOR = "origem"              # separa rótulos feitos às cegas dos validados a partir de uma sugestão (None para ignorar)
MAX_CHARS = 1500                      # trunca o texto para caber na janela junto com as opções de macro

# --- colunas da taxonomia ---
TAX_MACRO, TAX_SUB, TAX_DESC = "categoria_macro", "subcategoria", "descricao_gatilhos"
TAX_EXEMPLOS = "exemplos"             # opcional (None para ignorar): exemplos anexados à descrição
MAC_DESC = "descricao_macro"          # coluna de descrição na aba de macros

# --- Decider ---
URL = "http://127.0.0.1:8000/v1/systemone"
MODELO = "strands-decider-2B-qwen3.5-v1-2610"
JANELA_TOKENS = 4096                  # janela do modelo
ESTRATEGIA = "auto"                   # "auto" | "hierarquica" (macro → sub) | "plana" (todas as subs numa pergunta só)
LIMIAR_2A = 0.25                      # prob mínima (conjunta) para aceitar uma 2ª categoria
ESCALA_SENT = ["negativo", "neutro", "positivo"]

N_AMOSTRA = None                    # ex.: 30 para um teste rápido antes de rodar tudo

## 1. Leitura

In [ ]:
def ler(caminho, aba=0):
    caminho = str(caminho)
    return pd.read_excel(caminho, sheet_name=aba) if caminho.endswith((".xlsx", ".xls")) else pd.read_csv(caminho)

gold = ler(ARQ_GOLD, ABA_GOLD)
tax = ler(ARQ_TAXONOMIA, ABA_SUBS).dropna(subset=[TAX_MACRO])
macros_df = ler(ARQ_TAXONOMIA, ABA_MACROS) if ABA_MACROS else None

faltando = [c for c in [COL_ID, COL_ORDEM, COL_TEXTO, COL_MACRO, COL_SUB, COL_SENT] if c not in gold.columns]
faltando_tax = [c for c in [TAX_MACRO, TAX_SUB, TAX_DESC] if c not in tax.columns]
assert not faltando, f"Colunas do gold não encontradas: {faltando}. Colunas disponíveis: {list(gold.columns)}"
assert not faltando_tax, f"Colunas da taxonomia não encontradas: {faltando_tax}. Colunas disponíveis: {list(tax.columns)}"

# macro sem sub (ex.: Não Tabulável) → sub = nome da macro, nos DOIS lados, antes de converter para string
gold[COL_SUB] = gold[COL_SUB].fillna(gold[COL_MACRO])
tax[TAX_SUB] = tax[TAX_SUB].fillna(tax[TAX_MACRO])

# normaliza espaços e caixa do sentimento para a comparação não falhar por bobagem
for df, cols in [(gold, [COL_MACRO, COL_SUB, COL_SENT]), (tax, [TAX_MACRO, TAX_SUB])]:
    for c in cols:
        df[c] = df[c].astype(str).str.strip()
gold[COL_SENT] = gold[COL_SENT].str.lower()

# texto: trunca para caber na janela do Decider
gold[COL_TEXTO] = gold[COL_TEXTO].fillna("").astype(str)
n_trunc = (gold.drop_duplicates(COL_ID)[COL_TEXTO].str.len() > MAX_CHARS).sum()
gold[COL_TEXTO] = gold[COL_TEXTO].str.slice(0, MAX_CHARS)

tax[TAX_DESC] = tax[TAX_DESC].fillna("").astype(str).str.strip()
if TAX_EXEMPLOS:
    ex = tax[TAX_EXEMPLOS].fillna("").astype(str).str.strip()
    tax[TAX_DESC] = np.where(ex != "", tax[TAX_DESC] + " Exemplos: " + ex, tax[TAX_DESC])

print(f"gold: {len(gold)} linhas, {gold[COL_ID].nunique()} comentários ({n_trunc} truncados em {MAX_CHARS} caracteres)")
print(f"taxonomia: {tax[TAX_MACRO].nunique()} macros, {len(tax)} subcategorias")
if COL_ROTULADOR:
    print(gold.drop_duplicates(COL_ID)[COL_ROTULADOR].value_counts().to_string())

# subs do gold que não existem na taxonomia (o Decider nunca vai acertar essas)
pares_tax = set(zip(tax[TAX_MACRO], tax[TAX_SUB]))
fora = gold[[(m, s) not in pares_tax for m, s in zip(gold[COL_MACRO], gold[COL_SUB])]]
print(f"linhas do gold com (macro, sub) fora da taxonomia: {len(fora)}")
fora[[COL_MACRO, COL_SUB]].value_counts().head(20)

## 2. Mascaramento de dados sensíveis

Aplicado ao texto **antes** de qualquer outra etapa. Tudo por regex, nesta ordem (do mais específico ao mais genérico):

| Dado | Exemplo | Vira |
|---|---|---|
| E-mail | fulano@gmail.com | `[EMAIL]` |
| URL | https://... , www... | `[URL]` |
| Chave Pix aleatória (UUID) | 123e4567-e89b-... | `[CHAVE_PIX]` |
| Protocolo | "protocolo 2026101612345" | `[PROTOCOLO]` |
| Agência/conta | "ag 1234 conta 56789-0" | `[CONTA]` |
| CNPJ | 12.345.678/0001-90 | `[CNPJ]` |
| CPF | 123.456.789-00 ou 11 dígitos | `[CPF]` |
| Cartão | 13–19 dígitos, com ou sem espaço/traço | `[CARTAO]` |
| Telefone | (12) 99999-9999, +55 12 9..., 0800 | `[TELEFONE]` |
| CEP | 12345-678 | `[CEP]` |
| @usuário | @fulano_123 | `[USUARIO]` |
| Nome após "meu nome é / me chamo / sou o(a)" | "meu nome é Maria Silva" | `meu nome é [NOME]` |
| Sequência numérica longa restante | 6+ dígitos | `[NUMERO]` |

**Limite:** regex não pega nome solto no meio da frase ("falei com a Joana"), endereço por extenso ou dado de terceiros. A célula seguinte lista o que sobrou de suspeito para você revisar à mão.

In [ ]:
PADROES = [
    ("EMAIL",     r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    ("URL",       r"(?:https?://|www\.)\S+"),
    ("CHAVE_PIX", r"\b[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}\b"),
    ("PROTOCOLO", r"(?i)\b(?:protocolo|prot\.?|ticket|chamado)\s*(?:n[ºo°.]?\s*)?[:#\-]?\s*\d{4,}"),
    ("CONTA",     r"(?i)\b(?:ag[êe]ncia|ag\.?|conta(?:\s+corrente)?|c/c|cc)\s*(?:n[ºo°.]?\s*)?[:\-]?\s*\d[\d.\-]{2,}"),
    ("CNPJ",      r"\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b"),
    ("CPF",       r"\b\d{3}\.\d{3}\.\d{3}-?\d{2}\b|\b\d{9}-\d{2}\b|(?<!\d)\d{11}(?!\d)"),
    ("CARTAO",    r"(?<!\d)(?:\d[ -]?){12,18}\d(?!\d)"),
    ("TELEFONE",  r"(?:\+?55[\s-]?)?\(?\b\d{2}\)?[\s-]?9?\d{4}[\s-]?\d{4}\b|\b0800[\s-]?\d{3}[\s-]?\d{4}\b"),
    ("CEP",       r"\b\d{5}-\d{3}\b"),
    ("USUARIO",   r"(?<![\w@])@[A-Za-z0-9_.]{2,}"),
]
PADROES = [(nome, re.compile(p)) for nome, p in PADROES]
RE_NOME = re.compile(r"(?i)\b(meu nome [ée]|me chamo|sou (?:o|a))\s+([A-ZÀ-Ý][\wÀ-ÿ']+(?:\s+(?:d[aeo]s?\s+)?[A-ZÀ-Ý][\wÀ-ÿ']+){0,4})")
RE_NUMERO_LONGO = re.compile(r"(?<!\d)\d{6,}(?!\d)")

def mascarar(texto, contagem=None):
    if not isinstance(texto, str) or not texto:
        return texto
    t = unicodedata.normalize("NFKC", texto)
    for nome, rx in PADROES:
        t, n = rx.subn(f"[{nome}]", t)
        if contagem is not None and n:
            contagem[nome] = contagem.get(nome, 0) + n
    t, n = RE_NOME.subn(lambda m: f"{m.group(1)} [NOME]", t)
    if contagem is not None and n:
        contagem["NOME"] = contagem.get("NOME", 0) + n
    t, n = RE_NUMERO_LONGO.subn("[NUMERO]", t)
    if contagem is not None and n:
        contagem["NUMERO"] = contagem.get("NUMERO", 0) + n
    return t

# teste rápido com frases inventadas
for s in ["meu cpf é 123.456.789-00 e o cartão 5162 3412 9988 1234, liga no (12) 99876-5432",
          "manda no fulano.silva@gmail.com ou @fulano_123, protocolo 2026101612345",
          "Meu nome é Maria Aparecida Souza, ag 1234 conta 98765-4, CEP 12245-000"]:
    print(mascarar(s))

In [ ]:
contagem = {}
gold["texto_mask"] = gold[COL_TEXTO].map(lambda t: mascarar(t, contagem))

print("Mascaramentos aplicados:")
print(pd.Series(contagem, dtype=int).sort_values(ascending=False).to_string() if contagem else "nenhum")

# o que ainda parece sensível: dígitos soltos, palavras-gatilho, possíveis nomes próprios após preposição/verbo de contato
SUSPEITO = re.compile(r"\d{4,}|(?i:\b(?:rua|av\.?|avenida|endere[çc]o|bairro|nasc|rg)\b)|(?:\b(?:com|pra|para|a|o)\s+(?:sr\.?|sra\.?|atendente\s+)?[A-ZÀ-Ý][a-zà-ÿ]{2,}\b)")
suspeitos = gold.drop_duplicates(COL_ID)
suspeitos = suspeitos[suspeitos["texto_mask"].fillna("").str.contains(SUSPEITO)]
print(f"\nComentários para revisar à mão: {len(suspeitos)}")
suspeitos[[COL_ID, "texto_mask"]].head(30)

In [ ]:
# A partir daqui só o texto mascarado é usado. Removo a coluna original do DataFrame em memória
# e salvo uma cópia mascarada do gold (sem o texto original) para usar nas próximas rodadas.
gold = gold.drop(columns=[COL_TEXTO]).rename(columns={"texto_mask": COL_TEXTO})
gold.to_csv(PASTA_SAIDA / "gold_mascarado.csv", index=False)
print("salvo:", PASTA_SAIDA / "gold_mascarado.csv")

## 3. Cliente do Decider

O servidor recebe `state` (o texto) e um dicionário de perguntas. Cada pergunta tem um tipo:

- `choice`: escolhe 1 entre N opções. As opções vão em `criteria` como `{rótulo: descrição}`, e é aí que entram as descrições da taxonomia.
- `score`: nível numa escala ordenada (`criteria` é a lista de níveis, do menor ao maior).
- `noul`: sim/não.

O formato exato da resposta de `choice`/`score` não está documentado, então a função `distribuicao` procura a distribuição de probabilidades em formatos prováveis. **Rode a célula de teste e confira o JSON**: se ela não reconhecer o formato, o erro mostra a resposta e é só ajustar a função.

In [ ]:
sessao = requests.Session()

def decider(state, questions, tentativas=2):
    for i in range(tentativas):
        try:
            r = sessao.post(URL, json={"state": state, "model": MODELO, "questions": questions}, timeout=300)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if i == tentativas - 1:
                raise
            time.sleep(2)

def _num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)

def distribuicao(ans, opcoes):
    # Devolve {opção: prob} na ordem de `opcoes`.
    # choice: "probabilities" vem com o nome da opção como chave.
    # score:  "probabilities" vem com o índice ("0","1",...) e "legend" diz o nome de cada índice.
    opcoes = list(opcoes)
    probs = ans.get("probabilities")
    if not isinstance(probs, dict):
        raise ValueError("Resposta sem 'probabilities': " + json.dumps(ans, ensure_ascii=False)[:600])

    legenda = ans.get("legend")
    if isinstance(legenda, dict):
        probs = {legenda[k]: v for k, v in probs.items() if k in legenda}

    faltando = [o for o in opcoes if o not in probs]
    if faltando:
        raise ValueError(f"Opções sem probabilidade: {faltando}. Resposta: " + json.dumps(ans, ensure_ascii=False)[:600])
    return {o: float(probs[o]) for o in opcoes}

def confianca(ans):
    return next((float(ans[k]) for k in ("confidence", "conf") if k in ans and _num(ans[k])), None)

In [ ]:
# Teste com uma frase inventada. Confira o JSON: tem que vir uma probabilidade por opção.
teste_opcoes = {"Entrega › Atraso": "pedido chegou depois do prazo prometido ou ainda não chegou",
                "Entrega › Rastreio": "dúvida ou problema para acompanhar onde está o pedido",
                "Atendimento › Demora": "reclamação de tempo de espera no atendimento"}
resp = decider("comprei há 15 dias e até agora nada do pedido, o prazo era 5 dias úteis", {
    "assunto": {"type": "choice", "instructions": "Sobre o que o cliente está falando?", "criteria": teste_opcoes},
    "sent": {"type": "score", "instructions": "Qual o sentimento do cliente?", "criteria": ESCALA_SENT},
})
print(json.dumps(resp, indent=2, ensure_ascii=False))
print("\nassunto:", distribuicao(resp["answers"]["assunto"], teste_opcoes))
print("sentimento:", distribuicao(resp["answers"]["sent"], ESCALA_SENT))

## 4. Montagem das perguntas a partir da taxonomia

- **Macro**: a descrição da aba `Macros (Estágio 1)` (com as fronteiras entre macros) + a lista de nomes das subcategorias.
- **Sub**: cada subcategoria da macro com a **descrição completa**.
- **Sentimento**: *"Qual o sentimento do cliente em relação a «{sub}» ({descrição})?"*.

A célula de orçamento estima quantos tokens cada pergunta ocupa (≈ 1 token a cada 3,5 caracteres em português; é uma aproximação) e decide a estratégia quando `ESTRATEGIA = "auto"`.

In [ ]:
def rotulo(macro, sub):
    return f"{macro} › {sub}"

def resumo(txt, n=90):
    txt = " ".join(str(txt).split())
    return txt if len(txt) <= n else txt[:n].rsplit(" ", 1)[0] + "…"

# descrição da macro: a da aba de macros + a lista de subs; sem a aba, só a lista de subs
DESC_MACRO = dict(zip(macros_df.iloc[:, 0].astype(str).str.strip(), macros_df[MAC_DESC])) if macros_df is not None else {}
CRIT_MACRO = {m: (DESC_MACRO.get(m, "") + " Subcategorias: " if m in DESC_MACRO else "Inclui: ")
                 + "; ".join(g[TAX_SUB].astype(str))
              for m, g in tax.groupby(TAX_MACRO, sort=False)}
CRIT_SUB = {m: {rotulo(m, r[TAX_SUB]): (r[TAX_DESC] or r[TAX_SUB]) for _, r in g.iterrows()}
            for m, g in tax.groupby(TAX_MACRO, sort=False)}
CRIT_PLANA = {k: v for d in CRIT_SUB.values() for k, v in d.items()}
DESC_SUB = {rotulo(r[TAX_MACRO], r[TAX_SUB]): r[TAX_DESC] for _, r in tax.iterrows()}

PERG_ASSUNTO = "Sobre o que o cliente está falando neste comentário?"

def tokens(s):
    return int(len(s) / 3.5) + 1

def tamanho(criteria):
    return tokens(PERG_ASSUNTO) + sum(tokens(k) + tokens(v) + 4 for k, v in criteria.items())

tam_texto = gold.drop_duplicates(COL_ID)[COL_TEXTO].fillna("").map(tokens)
orc = pd.DataFrame({
    "pergunta": ["plana (todas as subs)", "macro"] + [f"sub · {m}" for m in CRIT_SUB],
    "tokens_opcoes": [tamanho(CRIT_PLANA), tamanho(CRIT_MACRO)] + [tamanho(c) for c in CRIT_SUB.values()],
})
orc["+ texto p95"] = orc["tokens_opcoes"] + int(tam_texto.quantile(.95))
orc["cabe?"] = orc["+ texto p95"] <= JANELA_TOKENS * 0.9   # 10% de folga para o template do modelo
print(f"texto: mediana {int(tam_texto.median())} tokens, p95 {int(tam_texto.quantile(.95))}, máx {tam_texto.max()}")

if ESTRATEGIA == "auto":
    ESTRATEGIA = "plana" if orc.loc[0, "cabe?"] else "hierarquica"
print("estratégia:", ESTRATEGIA)
orc

Se alguma linha `sub · <macro>` não couber, as descrições daquela macro estão longas demais: encurte-as (ou divida a macro) antes de rodar, senão o modelo trunca a entrada.

## 5. Classificação de um comentário

1. **Assunto**: na estratégia hierárquica, escolhe a macro; se a 2ª macro tiver probabilidade ≥ `LIMIAR_2A`, também olha as subs dela. A probabilidade de cada sub é **conjunta**: P(macro) × P(sub | macro), para dar para comparar subs de macros diferentes.
2. Fica a sub de maior probabilidade e, se houver, uma 2ª com probabilidade ≥ `LIMIAR_2A`.
3. **Sentimento** de cada categoria escolhida, condicionado ao tema, numa chamada só.

In [ ]:
def classificar(texto):
    aud = {}
    # --- assunto ---
    if ESTRATEGIA == "plana":
        r = decider(texto, {"sub": {"type": "choice", "instructions": PERG_ASSUNTO, "criteria": CRIT_PLANA}})
        p_sub = distribuicao(r["answers"]["sub"], CRIT_PLANA)
        conf_macro, aud["sub"] = None, r
    else:
        r = decider(texto, {"macro": {"type": "choice", "instructions": PERG_ASSUNTO, "criteria": CRIT_MACRO}})
        p_macro = distribuicao(r["answers"]["macro"], CRIT_MACRO)
        conf_macro, aud["macro"] = confianca(r["answers"]["macro"]), r
        ranking = sorted(p_macro, key=p_macro.get, reverse=True)
        macros = [ranking[0]] + ([ranking[1]] if len(ranking) > 1 and p_macro[ranking[1]] >= LIMIAR_2A else [])
        p_sub = {}
        for m in macros:
            if len(CRIT_SUB[m]) == 1:                       # macro com 1 sub só: não precisa perguntar
                p_sub[next(iter(CRIT_SUB[m]))] = p_macro[m]
                continue
            r = decider(texto, {"sub": {"type": "choice", "instructions": PERG_ASSUNTO, "criteria": CRIT_SUB[m]}})
            aud[f"sub · {m}"] = r
            for k, v in distribuicao(r["answers"]["sub"], CRIT_SUB[m]).items():
                p_sub[k] = p_macro[m] * v

    ranking = sorted(p_sub, key=p_sub.get, reverse=True)
    escolhidas = [ranking[0]] + ([ranking[1]] if len(ranking) > 1 and p_sub[ranking[1]] >= LIMIAR_2A else [])

    # --- sentimento condicionado ao tema ---
    perguntas = {f"sent_{i}": {"type": "score",
                               "instructions": f"Qual o sentimento do cliente em relação a «{k.split(' › ', 1)[1]}» ({resumo(DESC_SUB.get(k, ''), 120)})?",
                               "criteria": ESCALA_SENT}
                 for i, k in enumerate(escolhidas)}
    r = decider(texto, perguntas)
    aud["sentimento"] = r

    linhas = []
    for i, k in enumerate(escolhidas):
        ans = r["answers"][f"sent_{i}"]
        ps = distribuicao(ans, ESCALA_SENT)
        macro, sub = k.split(" › ", 1)
        linhas.append(dict(ordem=i + 1, macro=macro, subcategoria=sub,
                           prob=p_sub[k], conf_macro=conf_macro,
                           sentimento=max(ps, key=ps.get), p_sentimento=max(ps.values()),
                           **{f"p_{s}": ps[s] for s in ESCALA_SENT}))
    return linhas, aud

# teste em 1 comentário do gold
ex = gold.drop_duplicates(COL_ID).iloc[0]
print(ex[COL_TEXTO])
pd.DataFrame(classificar(ex[COL_TEXTO])[0])

## 6. Loop no gold

Salva o progresso a cada 25 comentários em `saida_decider/`; se parar no meio, rodar de novo continua de onde parou. Comece com `N_AMOSTRA = 30` na configuração para ver o tempo por comentário antes de rodar tudo.

In [ ]:
%pip install tqdm ipywidgets

In [ ]:
from tqdm.auto import tqdm

ARQ_PRED = PASTA_SAIDA / "decider_pred.csv"
ARQ_AUD = PASTA_SAIDA / "decider_auditoria.jsonl"
SALVAR_A_CADA = 25

textos = gold.drop_duplicates(COL_ID)[[COL_ID, COL_TEXTO]]
if N_AMOSTRA:
    textos = textos.sample(N_AMOSTRA, random_state=42)

pred = pd.read_csv(ARQ_PRED) if ARQ_PRED.exists() else pd.DataFrame()
feitos = set(pred["comment_id"].astype(str)) if len(pred) else set()
pendentes = textos[~textos[COL_ID].astype(str).isin(feitos)]
print(f"{len(feitos)} já feitos, {len(pendentes)} pendentes")

novas, n_erros = [], 0
barra = tqdm(pendentes.itertuples(index=False), total=len(pendentes), unit="coment")
with open(ARQ_AUD, "a", encoding="utf-8") as f_aud:
    for i, (cid, texto) in enumerate(barra, 1):
        t0 = time.perf_counter()
        try:
            linhas, aud = classificar(texto)
            for l in linhas:
                novas.append(dict(comment_id=cid, status="classificado", seg=time.perf_counter() - t0, **l))
            f_aud.write(json.dumps({"comment_id": str(cid), **aud}, ensure_ascii=False) + "\n")
        except Exception as e:
            n_erros += 1
            novas.append(dict(comment_id=cid, ordem=1, status="erro", erro=str(e)[:500]))
        barra.set_postfix(erros=n_erros, ultimo=f"{time.perf_counter() - t0:.1f}s")
        if i % SALVAR_A_CADA == 0 or i == len(pendentes):
            pred = pd.concat([pred, pd.DataFrame(novas)], ignore_index=True); novas = []
            pred.to_csv(ARQ_PRED, index=False)
            f_aud.flush()

pred = pd.read_csv(ARQ_PRED)
print(pred["status"].value_counts())
pred.head()

In [ ]:
# recria a tabela de comparação (1ª categoria: Decider × gold)
c = (pred[(pred.status == "classificado") & (pred.ordem == 1)]
     .assign(comment_id=lambda d: d.comment_id.astype(str))
     .merge(gold[gold[COL_ORDEM] == 1]
                .assign(comment_id=lambda d: d[COL_ID].astype(str))
                [["comment_id", COL_MACRO, COL_SUB]]
                .rename(columns={COL_MACRO: "macro_g", COL_SUB: "subcategoria_g"}),
            on="comment_id"))
c["acertou_macro"] = c.macro == c.macro_g
c["acertou_sub"] = c.subcategoria == c.subcategoria_g

# separa o erro em: errou a macro × acertou a macro mas errou a sub
c["etapa"] = np.where(~c.acertou_macro, "errou macro", np.where(~c.acertou_sub, "macro ok, errou sub", "acertou"))
print(c.etapa.value_counts(normalize=True).round(3))

# ele está "chutando sempre o mesmo"? (viés para poucas macros)
print("\nmacros mais escolhidas pelo Decider:\n", c.macro.value_counts(normalize=True).head(8).round(3))
print("\nmacros mais comuns no gold:\n", c.macro_g.value_counts(normalize=True).head(8).round(3))

# a confiança separa acerto de erro? (se sim, ainda serve como filtro do híbrido)
print("\nprob média quando acerta x erra a sub:\n", c.groupby("acertou_sub").prob.mean().round(3))

In [ ]:
# a macro do gold está entre as K primeiras do Decider?
aud = pd.read_json(ARQ_AUD, lines=True)
aud["comment_id"] = aud["comment_id"].astype(str)

def ranking_macros(resp):
    p = distribuicao(resp["answers"]["macro"], CRIT_MACRO)
    return sorted(p, key=p.get, reverse=True)

aud["ranking"] = aud["macro"].map(ranking_macros)
r = aud.merge(c[["comment_id", "macro_g"]], on="comment_id")
r["posicao"] = [rk.index(m) + 1 if m in rk else None for rk, m in zip(r.ranking, r.macro_g)]

for k in [1, 2, 3, 5]:
    print(f"macro do gold no top {k}: {(r.posicao <= k).mean():.0%}")
print("\nposição da macro certa nos erros:", sorted(r.loc[r.posicao > 1, "posicao"].astype(int).tolist()))

In [ ]:
r["situacao"] = np.where(r.posicao.isna(), "macro do gold fora da lista de opções",
                 np.where(r.posicao <= 3, "top 3", "posição > 3"))
print(r.situacao.value_counts().to_string(), "\n")

# os casos fora do top 3: qual era a certa e em que posição ficou
fora = r[r.situacao != "top 3"].merge(
    gold.drop_duplicates(COL_ID).assign(comment_id=lambda d: d[COL_ID].astype(str))[["comment_id", COL_TEXTO]],
    on="comment_id")
fora["top3_decider"] = fora.ranking.str[:3]
fora[["macro_g", "posicao", "top3_decider", COL_TEXTO]]

In [ ]:
fora.to_csv(PASTA_SAIDA / "fora_top3.csv", index=False)

## 7. Comparação com o gold

Três níveis de assunto, como combinado para avaliar o modelo, mais sentimento:

| Métrica | Definição |
|---|---|
| `macro_1a` | macro da 1ª categoria do Decider = macro da 1ª do gold |
| `sub_exata_1a` | sub da 1ª = sub da 1ª do gold |
| `sub_em_comum` | alguma sub do Decider está entre as subs do gold (até 2 de cada lado) |
| `sent_1a` | sentimento da 1ª categoria = sentimento da 1ª do gold |
| `sent_mesma_sub` | sentimento igual, só nos pares (comentário, sub) em que o Decider acertou a sub, ou seja, isola o erro de sentimento do erro de assunto |

**Lembrete sobre o gold:** se parte dos rótulos foi feita validando uma sugestão (de uma LLM, por exemplo) em vez de rotular às cegas, há viés de ancoragem e a acurácia tende a sair otimista para modelos que "pensam parecido" com quem sugeriu. Os rótulos feitos às cegas são a referência mais limpa; se `COL_ROTULADOR` estiver configurada, as métricas saem separadas.

In [ ]:
def kappa(a, b):
    a, b = pd.Series(a).astype(str).reset_index(drop=True), pd.Series(b).astype(str).reset_index(drop=True)
    po = (a == b).mean()
    pe = sum(a.value_counts(normalize=True).get(c, 0) * b.value_counts(normalize=True).get(c, 0) for c in set(a) | set(b))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan

g = gold.rename(columns={COL_ID: "comment_id", COL_ORDEM: "ordem", COL_MACRO: "macro", COL_SUB: "subcategoria", COL_SENT: "sentimento"})
g["comment_id"] = g["comment_id"].astype(str)
g["sentimento"] = g["sentimento"].str.lower()
p = pred[pred["status"] == "classificado"].copy()
p["comment_id"] = p["comment_id"].astype(str)

def avaliar(ids):
    g1 = g[(g.ordem == 1) & g.comment_id.isin(ids)].set_index("comment_id")
    p1 = p[(p.ordem == 1) & p.comment_id.isin(ids)].set_index("comment_id")
    comuns = g1.index.intersection(p1.index)
    g1, p1 = g1.loc[comuns], p1.loc[comuns]
    subs_g = g[g.comment_id.isin(comuns)].groupby("comment_id").subcategoria.apply(set)
    subs_p = p[p.comment_id.isin(comuns)].groupby("comment_id").subcategoria.apply(set)
    mesma = p.merge(g, on=["comment_id", "subcategoria"], suffixes=("", "_g"))
    mesma = mesma[mesma.comment_id.isin(comuns)]
    return pd.Series({
        "n": len(comuns),
        "macro_1a": (g1.macro == p1.macro).mean(),
        "kappa_macro": kappa(g1.macro, p1.macro),
        "sub_exata_1a": (g1.subcategoria == p1.subcategoria).mean(),
        "kappa_sub": kappa(g1.subcategoria, p1.subcategoria),
        "sub_em_comum": np.mean([bool(subs_g[i] & subs_p[i]) for i in comuns]),
        "sent_1a": (g1.sentimento == p1.sentimento).mean(),
        "kappa_sent": kappa(g1.sentimento, p1.sentimento),
        "sent_mesma_sub": (mesma.sentimento == mesma.sentimento_g).mean() if len(mesma) else np.nan,
        "n_mesma_sub": len(mesma),
    })

avaliados = sorted(set(p.comment_id))
res = {"todos": avaliar(avaliados)}
if COL_ROTULADOR:
    for grupo, ids in g[g.comment_id.isin(avaliados)].groupby(COL_ROTULADOR).comment_id:
        res[str(grupo)] = avaliar(set(ids))
resultado = pd.DataFrame(res)
print(f"erros de execução: {(pred.status == 'erro').sum()}")
resultado.style.format(lambda v: f"{v:.0f}" if v >= 2 else f"{v:.1%}" if v == v else "–")

### Calibração: o Decider sabe quando está errando?

Acerto da sub (1ª categoria) por faixa de probabilidade. Se a acurácia sobe com a probabilidade, dá para montar o **híbrido**: aceitar o Decider acima de um limiar e mandar o resto para a LLM. A última coluna mostra quanto do volume ficaria com o Decider em cada limiar.

In [ ]:
c = p[p.ordem == 1].merge(g[g.ordem == 1][["comment_id", "macro", "subcategoria"]], on="comment_id", suffixes=("", "_g"))
c["acertou_sub"] = c.subcategoria == c.subcategoria_g
c["acertou_macro"] = c.macro == c.macro_g
c["faixa"] = pd.cut(c.prob, [0, .2, .4, .6, .8, 1.0], include_lowest=True)
calib = c.groupby("faixa", observed=True).agg(n=("acertou_sub", "size"), acc_sub=("acertou_sub", "mean"), acc_macro=("acertou_macro", "mean"))

limiares = pd.DataFrame([{"limiar": t, "% volume no Decider": (c.prob >= t).mean(),
                          "acc_sub acima do limiar": c.loc[c.prob >= t, "acertou_sub"].mean()}
                         for t in [.3, .4, .5, .6, .7, .8, .9]])
display(calib.style.format({"acc_sub": "{:.1%}", "acc_macro": "{:.1%}"}))
display(limiares.style.format({"% volume no Decider": "{:.1%}", "acc_sub acima do limiar": "{:.1%}"}))

### Onde ele erra

Matriz de confusão de macro e as subs com mais erro. Os erros vão para `saida_decider/erros.csv`, com o texto mascarado, para leitura.

In [ ]:
display(pd.crosstab(c.macro_g, c.macro, rownames=["gold"], colnames=["decider"], margins=True))

por_sub = (c.groupby("subcategoria_g")
             .agg(n=("acertou_sub", "size"), acc=("acertou_sub", "mean"),
                  mais_confundida_com=("subcategoria", lambda s: s.value_counts().index[0]))
             .query("n >= 3").sort_values("acc"))
display(por_sub.head(20).style.format({"acc": "{:.0%}"}))

sent = p[p.ordem == 1].merge(g[g.ordem == 1][["comment_id", "sentimento"]], on="comment_id", suffixes=("", "_g"))
display(pd.crosstab(sent.sentimento_g, sent.sentimento, rownames=["gold"], colnames=["decider"], margins=True))

erros = c[~c.acertou_sub].merge(gold.drop_duplicates(COL_ID)[[COL_ID, COL_TEXTO]].assign(**{COL_ID: lambda d: d[COL_ID].astype(str)}),
                                left_on="comment_id", right_on=COL_ID)
erros[["comment_id", COL_TEXTO, "macro_g", "subcategoria_g", "macro", "subcategoria", "prob"]] \
    .sort_values("prob", ascending=False).to_csv(PASTA_SAIDA / "erros.csv", index=False)
print("erros salvos em", PASTA_SAIDA / "erros.csv")

## 8. Próximos passos

- **Comparar com uma LLM** (ex.: `gpt-4o-mini`) nos mesmos comentários: gerar a saída no mesmo formato longo e rodar a função `avaliar` nela.
- **Ajustar o `LIMIAR_2A`** olhando `sub_em_comum` e o número médio de categorias por comentário contra o gold.
- **Testar um modelo maior** se o 2B ficar fraco em português: o `strands-decider-E4B-gemma4-v1-2610` (precisa instalar do GitHub, ver o README) mudando só `MODELO` e o comando do servidor.
- **Descrições**: as subs com pior acurácia na tabela acima são candidatas a reescrever a descrição, o que também ajuda a RAG.